In [0]:
from pyspark.sql.functions import *

In [0]:
import sys
import importlib
sys.path.append('/Workspace/Users/kiranmanam9490@outlook.com/customer360')

import common.configuration
importlib.reload(common.configuration)
common.configuration.dbutils = dbutils

from common.configuration import Configuration

conf = Configuration()

In [0]:
df = spark.read.format('parquet').load(conf.base_url + '/bronze/payments')


In [0]:
df = df.withColumn(
    "payment_date",
    coalesce(
        try_to_date(col("payment_date"), "yyyy/MM/dd"),
        try_to_date(col("payment_date"), "dd-MM-yyyy"),
        try_to_date(col("payment_date"), "yyyy-MM-dd"),
        try_to_date(col("payment_date"), "yyyyMMdd"),
        try_to_date(col("payment_date"), "MM-dd-yyyy"),
        try_to_date(col("payment_date"), "dd/MM/yyyy"),
    )
)

In [0]:
df =df.withColumn('payment_method',initcap(col("payment_method")))\
    .withColumn('payment_method',when(col('payment_method')=='CreditCard','Credit Card').when(col('payment_method')=='DebitCard','Debit Card').otherwise(col('payment_method')))

df = df.withColumn('payment_status',initcap(when(col("payment_status").isNull(),"Failed").otherwise(col("payment_status"))))

df = df.withColumn("amount",col("amount").cast("double"))
df = df.withColumn('amount',when(col('amount').isNull(),0).when(col('amount')<0,0).otherwise(col('amount')))



In [0]:
df =df.dropDuplicates(["payment_id","customer_id"])
df = df.dropna(subset=["payment_id","customer_id"])

In [0]:
df = df.withColumn("customer_id", col("customer_id").cast("int"))

In [0]:
df.write.format("delta").mode("overwrite").save(conf.base_url + '/silver/payments')